# 06 ? Onion Price Model Evaluation

## Project objective
Evaluate the trained multi-market onion price prediction models using their saved test predictions.

This notebook:
- Checks the saved prediction and model-comparison files.
- Recalculates MAE, RMSE, and R? independently.
- Compares recalculated metrics with the training report.
- Examines prediction errors and residuals.
- Creates evaluation charts and CSV reports.

**Important:** This notebook evaluates existing predictions. It does not retrain models or change the test dataset.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# Locate the project root from the current notebook execution directory.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

PROJECT_DIR = next(
    (
        folder.resolve()
        for folder in candidates
        if (folder / "reports/onion_multi_market_test_predictions.csv").exists()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project prediction report. "
        "Run this notebook from the project or notebooks directory."
    )

PREDICTION_FILE = PROJECT_DIR / "reports/onion_multi_market_test_predictions.csv"
COMPARISON_FILE = PROJECT_DIR / "reports/onion_multi_market_model_comparison.csv"
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Prediction file exists:", PREDICTION_FILE.exists())
print("Comparison file exists:", COMPARISON_FILE.exists())
print("Figure directory:", FIGURE_DIR)


## 1. Load the saved evaluation data

The prediction file contains the actual test price, predicted price, model name, market information, and arrival date. The comparison report contains the metrics calculated during model training.

In [ ]:
predictions = pd.read_csv(PREDICTION_FILE)
comparison = pd.read_csv(COMPARISON_FILE)

required_prediction_columns = {
    "State", "District", "Market", "Variety", "Grade",
    "Arrival_Date", "Actual_Price", "Predicted_Price", "Model"
}
required_comparison_columns = {
    "Model", "MAE", "RMSE", "R2", "Training_Seconds", "Status"
}

missing_predictions = required_prediction_columns - set(predictions.columns)
missing_comparison = required_comparison_columns - set(comparison.columns)

if missing_predictions:
    raise ValueError(f"Missing prediction columns: {sorted(missing_predictions)}")
if missing_comparison:
    raise ValueError(f"Missing comparison columns: {sorted(missing_comparison)}")

predictions["Arrival_Date"] = pd.to_datetime(
    predictions["Arrival_Date"], errors="coerce"
)

predictions["Actual_Price"] = pd.to_numeric(
    predictions["Actual_Price"], errors="coerce"
)
predictions["Predicted_Price"] = pd.to_numeric(
    predictions["Predicted_Price"], errors="coerce"
)

predictions = predictions.dropna(
    subset=["Actual_Price", "Predicted_Price", "Model"]
).copy()

predictions["Residual"] = (
    predictions["Actual_Price"] - predictions["Predicted_Price"]
)
predictions["Absolute_Error"] = predictions["Residual"].abs()

print("Prediction records:", f"{len(predictions):,}")
print("Models:", predictions["Model"].nunique())
print("Unique markets:", predictions["Market"].nunique())
print("Date range:", predictions["Arrival_Date"].min(), "to", predictions["Arrival_Date"].max())

display(comparison)


## 2. Check prediction data integrity

Each model should have predictions for the same test records. We check the number of rows, missing values, duplicate records, and the actual test-price range.

In [ ]:
model_counts = (
    predictions.groupby("Model")
    .agg(
        Prediction_Records=("Actual_Price", "size"),
        Unique_Markets=("Market", "nunique"),
        Unique_States=("State", "nunique"),
        Mean_Actual_Price=("Actual_Price", "mean"),
        Mean_Predicted_Price=("Predicted_Price", "mean")
    )
    .reset_index()
)

print("Missing values in required prediction fields:")
display(
    predictions[
        ["Actual_Price", "Predicted_Price", "Model", "Arrival_Date"]
    ].isna().sum().to_frame("Missing_Count")
)

print("Prediction records by model:")
display(model_counts)

print("Duplicate rows in prediction report:", int(predictions.duplicated().sum()))


## 3. Recalculate model evaluation metrics

We independently calculate:

- **MAE:** average absolute prediction error. Lower is better.
- **RMSE:** square root of average squared prediction error. Lower is better and large errors receive greater weight.
- **R?:** proportion of target-price variation explained by the predictions. Higher is generally better.

The previous-price baseline is included as a reference model.

In [ ]:
metric_rows = []

for model_name, group in predictions.groupby("Model", sort=False):
    actual = group["Actual_Price"].to_numpy()
    predicted = group["Predicted_Price"].to_numpy()

    metric_rows.append({
        "Model": model_name,
        "Records": len(group),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    })

recalculated_metrics = pd.DataFrame(metric_rows)

recalculated_metrics = recalculated_metrics.sort_values(
    "RMSE", ascending=True
).reset_index(drop=True)

display(
    recalculated_metrics.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.5f}"
    })
)


## 4. Verify the saved model comparison report

The recalculated results are compared with the metrics saved during model training. Small differences may occur due to rounding in the saved report.

In [ ]:
verification = recalculated_metrics.merge(
    comparison[["Model", "MAE", "RMSE", "R2", "Training_Seconds", "Status"]],
    on="Model",
    how="left",
    suffixes=("_Recalculated", "_Saved")
)

for metric in ["MAE", "RMSE", "R2"]:
    verification[f"{metric}_Difference"] = (
        verification[f"{metric}_Recalculated"]
        - verification[f"{metric}_Saved"]
    )

verification["Metrics_Match"] = (
    verification[[
        "MAE_Difference", "RMSE_Difference", "R2_Difference"
    ]].abs().max(axis=1) < 0.02
)

display(
    verification[[
        "Model",
        "MAE_Recalculated", "MAE_Saved",
        "RMSE_Recalculated", "RMSE_Saved",
        "R2_Recalculated", "R2_Saved",
        "Training_Seconds", "Status", "Metrics_Match"
    ]]
)

print(
    "All model metrics verified:",
    bool(verification["Metrics_Match"].all())
)


## 5. Compare model performance

These charts compare the three main evaluation metrics. The best model can differ depending on which metric matters most.

In [ ]:
plot_metrics = recalculated_metrics.set_index("Model")

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

for ax, metric, title, color in [
    (axes[0], "MAE", "Mean Absolute Error (lower is better)", "steelblue"),
    (axes[1], "RMSE", "Root Mean Squared Error (lower is better)", "darkorange"),
    (axes[2], "R2", "R-squared (higher is better)", "seagreen")
]:
    values = plot_metrics[metric].sort_values(
        ascending=(metric != "R2")
    )
    ax.barh(values.index, values.values, color=color)
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "onion_model_evaluation_metrics.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 6. Analyse prediction residuals

A residual is calculated as:

**Residual = Actual Price ? Predicted Price**

- A positive residual means the model underestimated the actual price.
- A negative residual means the model overestimated the actual price.
- Residuals close to zero indicate more accurate predictions.

In [ ]:
residual_summary = (
    predictions.groupby("Model")
    .agg(
        Mean_Residual=("Residual", "mean"),
        Median_Residual=("Residual", "median"),
        Residual_Std=("Residual", "std"),
        Mean_Absolute_Error=("Absolute_Error", "mean"),
        Maximum_Absolute_Error=("Absolute_Error", "max")
    )
    .reset_index()
)

display(residual_summary.style.format({
    "Mean_Residual": "{:.2f}",
    "Median_Residual": "{:.2f}",
    "Residual_Std": "{:.2f}",
    "Mean_Absolute_Error": "{:.2f}",
    "Maximum_Absolute_Error": "{:.2f}"
}))


In [ ]:
# Compare residual distributions across models.
model_names = list(predictions["Model"].drop_duplicates())

fig, ax = plt.subplots(figsize=(12, 6))
residual_groups = [
    predictions.loc[predictions["Model"] == name, "Residual"].sample(
        n=min(5000, (predictions["Model"] == name).sum()),
        random_state=42
    ).to_numpy()
    for name in model_names
]

ax.boxplot(residual_groups, tick_labels=model_names, showfliers=False)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Residual Distribution by Model")
ax.set_ylabel("Residual (Actual ? Predicted)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "onion_model_evaluation_residual_boxplot.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 7. Actual versus predicted prices

The scatter plot checks how closely predictions follow actual prices. Points closer to the diagonal reference line indicate more accurate predictions.

In [ ]:
selected_model = "Gradient Boosting"
selected = predictions[predictions["Model"] == selected_model].copy()

if selected.empty:
    selected_model = recalculated_metrics.iloc[0]["Model"]
    selected = predictions[predictions["Model"] == selected_model].copy()

scatter_sample = selected.sample(
    n=min(5000, len(selected)),
    random_state=42
)

low = min(
    scatter_sample["Actual_Price"].min(),
    scatter_sample["Predicted_Price"].min()
)
high = max(
    scatter_sample["Actual_Price"].max(),
    scatter_sample["Predicted_Price"].max()
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    scatter_sample["Actual_Price"],
    scatter_sample["Predicted_Price"],
    alpha=0.35,
    s=12
)
ax.plot([low, high], [low, high], linestyle="--", linewidth=2)
ax.set_title(f"Actual vs Predicted Prices ? {selected_model}")
ax.set_xlabel("Actual Price")
ax.set_ylabel("Predicted Price")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "onion_actual_vs_predicted.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 8. Examine the largest prediction errors

The following table identifies observations with the largest absolute prediction errors. These records can help identify unusual market movements, volatile prices, or difficult-to-predict observations.

In [ ]:
largest_errors = (
    predictions.sort_values("Absolute_Error", ascending=False)
    [[
        "State", "District", "Market", "Variety", "Grade",
        "Arrival_Date", "Actual_Price", "Predicted_Price",
        "Residual", "Absolute_Error", "Model"
    ]]
    .head(25)
    .reset_index(drop=True)
)

display(largest_errors)


## 9. Evaluation by state

State-level MAE is calculated to check whether prediction errors vary across locations. States with fewer test observations should be interpreted cautiously.

In [ ]:
state_metrics = (
    predictions.groupby(["Model", "State"])
    .apply(
        lambda group: pd.Series({
            "Records": len(group),
            "MAE": mean_absolute_error(
                group["Actual_Price"], group["Predicted_Price"]
            ),
            "RMSE": np.sqrt(mean_squared_error(
                group["Actual_Price"], group["Predicted_Price"]
            ))
        }),
        include_groups=False
    )
    .reset_index()
)

display(
    state_metrics.sort_values(
        ["Model", "MAE"], ascending=[True, False]
    ).head(30)
)


## 10. Save evaluation results

The evaluation metrics, residual summaries, state-level results, and largest prediction errors are saved as CSV files for further analysis and project documentation.

In [ ]:
recalculated_metrics.to_csv(
    REPORT_DIR / "onion_model_evaluation_metrics.csv", index=False
)
verification.to_csv(
    REPORT_DIR / "onion_model_evaluation_verification.csv", index=False
)
residual_summary.to_csv(
    REPORT_DIR / "onion_model_residual_summary.csv", index=False
)
state_metrics.to_csv(
    REPORT_DIR / "onion_model_state_evaluation.csv", index=False
)
largest_errors.to_csv(
    REPORT_DIR / "onion_model_largest_prediction_errors.csv", index=False
)

print("Evaluation reports saved:")
for name in [
    "onion_model_evaluation_metrics.csv",
    "onion_model_evaluation_verification.csv",
    "onion_model_residual_summary.csv",
    "onion_model_state_evaluation.csv",
    "onion_model_largest_prediction_errors.csv"
]:
    print("-", REPORT_DIR / name)

print("\nEvaluation charts saved:")
for name in [
    "onion_model_evaluation_metrics.png",
    "onion_model_evaluation_residual_boxplot.png",
    "onion_actual_vs_predicted.png"
]:
    print("-", FIGURE_DIR / name)


## 11. Final evaluation findings

The saved comparison report indicates that:

- The **Previous Price Baseline** has the lowest MAE, so it has the smallest average absolute error in this test set.
- **Gradient Boosting** has the lowest RMSE and the highest R? among the compared models.
- The machine-learning models improve RMSE and R? over the previous-price baseline, but they do not improve MAE in this evaluation.
- No single model is best across every metric. Model selection should depend on the business cost of different types of prediction errors.

These findings apply to the saved test period and market records. They should not be treated as a guarantee of future performance.

**Result:** Model evaluation, metric verification, residual analysis, and report generation are completed using the existing test predictions. No model was retrained in this notebook.
